In [ ]:
# Setup otomatis untuk Google Colab
# Cell ini menyinkronkan file notebook dengan repository GitHub agar data CSV ikut tersedia.

import os
from pathlib import Path

REPO_URL = 'https://github.com/AchmadAlvin/UAS_DATAENGINEERING.git'
REPO_DIR = Path('/content/UAS_DATAENGINEERING')

if Path('/content').exists():
    if REPO_DIR.exists():
        %cd /content/UAS_DATAENGINEERING
        !git pull
    else:
        %cd /content
        !git clone {REPO_URL}
        %cd /content/UAS_DATAENGINEERING

    DATA_DIR = Path('data/raw')
else:
    DATA_DIR = Path('../data/raw')

print(f'Data folder: {DATA_DIR.resolve() if DATA_DIR.exists() else DATA_DIR}')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 1
print("=" * 65)
print("  BAGIAN 1 — IMPLEMENTASI PEMBACAAN DATA")
print("=" * 65)

# 1A. Harga Minyak Mentah Global (WTI)
DATA_DIR_CANDIDATES = [
    Path('data/raw'),
    Path('../data/raw'),
    Path('/content/UAS_DATAENGINEERING/data/raw'),
    Path('/content'),
]
DATA_DIR = next((path for path in DATA_DIR_CANDIDATES if path.exists()), DATA_DIR_CANDIDATES[0])

print("\n[1A] Membaca crude-oil-price.csv ...")
df_oil_raw = pd.read_csv(DATA_DIR / 'crude-oil-price.csv')
df_oil_raw['date'] = pd.to_datetime(df_oil_raw['date'], utc=True).dt.tz_localize(None)
df_oil_raw.rename(columns={'price': 'harga_minyak', 'percentChange': 'pct_change_minyak'}, inplace=True)
# Filter 10 tahun: 2016–2026
df_oil = df_oil_raw[df_oil_raw['date'].dt.year.between(2016, 2026)].copy()
df_oil['tahun'] = df_oil['date'].dt.year
df_oil['bulan'] = df_oil['date'].dt.month
df_oil = df_oil[['date','tahun','bulan','harga_minyak','pct_change_minyak','change']].reset_index(drop=True)
print(f"     Berhasil | {len(df_oil)} baris | "
      f"{df_oil['date'].min().strftime('%b %Y')} – {df_oil['date'].max().strftime('%b %Y')}")

# 1B. Kurs USD/IDR
print("\n[1B] Membaca USD_IDR_Historical_Data.csv ...")
df_kurs_raw = pd.read_csv(DATA_DIR / 'USD_IDR Historical Data (1).csv')
df_kurs_raw['date'] = pd.to_datetime(df_kurs_raw['Date'], format='%m/%d/%Y')
df_kurs_raw['kurs_usd_idr'] = df_kurs_raw['Price'].str.replace(',','').astype(float)
df_kurs_raw['pct_change_kurs'] = df_kurs_raw['Change %'].str.replace('%','').astype(float)
df_kurs = df_kurs_raw[['date','kurs_usd_idr','pct_change_kurs']].sort_values('date').reset_index(drop=True)
df_kurs['tahun'] = df_kurs['date'].dt.year
df_kurs['bulan'] = df_kurs['date'].dt.month
print(f"     Berhasil | {len(df_kurs)} baris | "
      f"{df_kurs['date'].min().strftime('%b %Y')} – {df_kurs['date'].max().strftime('%b %Y')}")

# 1C. Inflasi Bulanan BPS
print("\n[1C] Membaca Inflasi BPS 2020–2026 ...")
bulan_map = {
    'Januari':1,'Februari':2,'Maret':3,'April':4,'Mei':5,'Juni':6,
    'Juli':7,'Agustus':8,'September':9,'Oktober':10,'November':11,'Desember':12
}
records = []
years = range(2020, 2027)

for year in years:
    fname = DATA_DIR / f"Inflasi Bulanan (M-to-M), {year}.csv"
    try:
        df_raw = pd.read_csv(fname, header=None, skiprows=4, encoding='utf-8-sig', sep=None, engine='python')
        df_raw.columns = ['kota'] + list(bulan_map.keys()) + ['Tahunan']
        df_id = df_raw[df_raw['kota'].str.strip().str.upper() == 'INDONESIA']

        if not df_id.empty:
            for nm, nb in bulan_map.items():
                val = df_id[nm].values
                if len(val):
                    try: v = float(str(val[0]).replace(',','.'))
                    except: v = np.nan
                    records.append({'tahun':year,'bulan':nb,'nama_bulan':nm,'inflasi_mtom':v})
            print(f"     {year} OK")
    except Exception as e:
        print(f"     {year} ERROR: {e}")

df_inf = pd.DataFrame(records)
df_inf['date'] = pd.to_datetime(
    df_inf[['tahun','bulan']].assign(day=1).rename(columns={'tahun':'year','bulan':'month'})
)
df_inf = df_inf[df_inf['inflasi_mtom'].notna()].sort_values('date').reset_index(drop=True)
print(f"     Total: {len(df_inf)} baris | "
      f"{df_inf['date'].min().strftime('%b %Y')} – {df_inf['date'].max().strftime('%b %Y')}")

# 1D. Data BBM Ritel (Hardcode terverifikasi)
print("\n[1D] Membangun data BBM Ritel (hardcode dari Pertamina/Katadata) ...")
bbm_records = []
for year in range(2020, 2027):
    for month in range(1, 13):
        d = pd.Timestamp(year=year, month=month, day=1)
        if d > pd.Timestamp('2026-03-01'): break
        if d >= pd.Timestamp('2022-09-01'):
            bbm_records.append({'date':d,'harga_pertalite':10000,'harga_solar':6800,'policy_change':1 if d==pd.Timestamp('2022-09-01') else 0})
        else:
            bbm_records.append({'date':d,'harga_pertalite':7650,'harga_solar':5150,'policy_change':0})
df_bbm = pd.DataFrame(bbm_records)
print(f"     {len(df_bbm)} baris | Policy_Change=1 pada: Sep 2022")

  BAGIAN 1 — IMPLEMENTASI PEMBACAAN DATA

[1A] Membaca crude-oil-price.csv ...
     Berhasil | 123 baris | Jan 2016 – Mar 2026

[1B] Membaca USD_IDR_Historical_Data.csv ...
     Berhasil | 75 baris | Jan 2020 – Mar 2026

[1C] Membaca Inflasi BPS 2020–2026 ...
     2020 OK
     2021 OK
     2022 OK
     2023 OK
     2024 OK
     2025 OK
     2026 OK
     Total: 74 baris | Jan 2020 – Feb 2026

[1D] Membangun data BBM Ritel (hardcode dari Pertamina/Katadata) ...
     75 baris | Policy_Change=1 pada: Sep 2022


In [ ]:
# ══════════════════════════════════════════════════════════
# BAGIAN 2 — ANALISIS STRUKTUR DATA
# ══════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("  BAGIAN 2 — ANALISIS STRUKTUR DATA")
print("=" * 65)

datasets = {
    'Harga Minyak Global (WTI)': df_oil,
    'Kurs USD/IDR': df_kurs,
    'Inflasi BPS M-to-M': df_inf,
    'Harga BBM Ritel': df_bbm,
}

for nama, df in datasets.items():
    print(f"\n== {nama} ==")
    print(f"  Shape     : {df.shape[0]} baris × {df.shape[1]} kolom")
    print(f"  Tipe Data :")
    for col, dtype in df.dtypes.items():
        print(f"    {col:<25} {dtype}")
    print(f"  Missing   : {df.isnull().sum().sum()} total")


  BAGIAN 2 — ANALISIS STRUKTUR DATA

== Harga Minyak Global (WTI) ==
  Shape     : 123 baris × 6 kolom
  Tipe Data :
    date                      datetime64[ns]
    tahun                     int32
    bulan                     int32
    harga_minyak              float64
    pct_change_minyak         float64
    change                    float64
  Missing   : 0 total

== Kurs USD/IDR ==
  Shape     : 75 baris × 5 kolom
  Tipe Data :
    date                      datetime64[ns]
    kurs_usd_idr              float64
    pct_change_kurs           float64
    tahun                     int32
    bulan                     int32
  Missing   : 0 total

== Inflasi BPS M-to-M ==
  Shape     : 74 baris × 5 kolom
  Tipe Data :
    tahun                     int64
    bulan                     int64
    nama_bulan                object
    inflasi_mtom              float64
    date                      datetime64[ns]
  Missing   : 0 total

== Harga BBM Ritel ==
  Shape     : 75 baris × 4 kolom
  Ti

In [ ]:

# ══════════════════════════════════════════════════════════
# BAGIAN 3 — DATA PROFILING
# ══════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("  BAGIAN 3 — DATA PROFILING")
print("=" * 65)

def profile_numerik(df, kolom_list, nama):
    print(f"\n── Profiling: {nama} ──")
    df_num = df[kolom_list].copy()
    stats_df = pd.DataFrame({
        'Count':    df_num.count(),
        'Missing':  df_num.isnull().sum(),
        'Mean':     df_num.mean().round(3),
        'Std':      df_num.std().round(3),
        'Min':      df_num.min().round(3),
        'Q25':      df_num.quantile(0.25).round(3),
        'Median':   df_num.median().round(3),
        'Q75':      df_num.quantile(0.75).round(3),
        'Max':      df_num.max().round(3),
        'Skewness': df_num.skew().round(3),
    })
    print(stats_df.to_string())

profile_numerik(df_oil,  ['harga_minyak','pct_change_minyak'], 'Harga Minyak WTI (2016–2026)')
profile_numerik(df_kurs, ['kurs_usd_idr','pct_change_kurs'],   'Kurs USD/IDR (2020–2026)')
profile_numerik(df_inf,  ['inflasi_mtom'],                     'Inflasi M-to-M Indonesia (2020–2026)')
profile_numerik(df_bbm,  ['harga_pertalite','harga_solar'],    'Harga BBM Ritel')


  BAGIAN 3 — DATA PROFILING

── Profiling: Harga Minyak WTI (2016–2026) ──
                   Count  Missing    Mean     Std     Min     Q25  Median     Q75      Max  Skewness
harga_minyak         123        0  63.562  16.686  18.840  51.615   64.01  74.955  111.910     0.106
pct_change_minyak    123        0   1.345  12.674 -54.245  -5.539    1.80   7.073   88.376     1.938

── Profiling: Kurs USD/IDR (2020–2026) ──
                 Count  Missing       Mean      Std       Min        Q25    Median       Q75       Max  Skewness
kurs_usd_idr        75        0  15291.303  900.093  13650.00  14495.000  15135.00  16240.00  16929.70     0.234
pct_change_kurs     75        0      0.297    2.551     -9.05     -0.625      0.21      1.42     13.67     1.196

── Profiling: Inflasi M-to-M Indonesia (2020–2026) ──
              Count  Missing   Mean    Std   Min    Q25  Median    Q75   Max  Skewness
inflasi_mtom     74        0  0.224  0.367 -0.76 -0.008   0.175  0.378  1.65     0.999

── Profil

In [ ]:

# ══════════════════════════════════════════════════════════
# BAGIAN 4 — IMPLEMENTASI STRUKTUR DATA (MERGE)
# ══════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("  BAGIAN 4 — IMPLEMENTASI STRUKTUR DATA")
print("=" * 65)

# Gabungkan semua dataset (inner join pada periode overlap)
df_master = df_inf[['date','tahun','bulan','nama_bulan','inflasi_mtom']].copy()
df_master = df_master.merge(
    df_oil[['date','harga_minyak','pct_change_minyak']],
    on='date', how='inner'
)
df_master = df_master.merge(
    df_kurs[['date','kurs_usd_idr','pct_change_kurs']],
    on='date', how='left'
)
df_master = df_master.merge(
    df_bbm[['date','harga_pertalite','harga_solar','policy_change']],
    on='date', how='left'
)
df_master = df_master.sort_values('date').reset_index(drop=True)

print(f"\n Dataset Master: {df_master.shape[0]} baris × {df_master.shape[1]} kolom")
print(f"  Periode: {df_master['date'].min().strftime('%b %Y')} – {df_master['date'].max().strftime('%b %Y')}")
print(f"\nRingkasan per Tahun:")
summary = df_master.groupby('tahun').agg(
    n_bulan=('inflasi_mtom','count'),
    rata_inflasi=('inflasi_mtom','mean'),
    rata_minyak=('harga_minyak','mean'),
    rata_kurs=('kurs_usd_idr','mean'),
).round(3)
print(summary.to_string())

# Dictionary struktur per tahun
struktur = {}
for year, grp in df_master.groupby('tahun'):
    struktur[year] = {
        'data': grp.reset_index(drop=True),
        'n': len(grp),
        'inflasi_mean': grp['inflasi_mtom'].mean(),
        'inflasi_max_bulan': grp.loc[grp['inflasi_mtom'].idxmax(), 'nama_bulan'],
        'minyak_mean': grp['harga_minyak'].mean(),
    }


  BAGIAN 4 — IMPLEMENTASI STRUKTUR DATA

 Dataset Master: 74 baris × 12 kolom
  Periode: Jan 2020 – Feb 2026

Ringkasan per Tahun:
       n_bulan  rata_inflasi  rata_minyak  rata_kurs
tahun                                               
2020        12         0.142       38.596  14545.833
2021        12         0.155       68.069  14313.333
2022        12         0.450       92.289  14905.667
2023        12         0.215       77.437  15199.167
2024        12         0.130       75.499  15888.750
2025        12         0.243       64.206  16511.250
2026         2         0.265       66.115  16775.000
